<a href="https://colab.research.google.com/github/sudhars97/Forum-Posts-code/blob/main/DL_M2_RR_C7_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import yfinance as ticker
from sklearn.preprocessing import StandardScaler

# 1. DATA ACQUISITION (No synthetic data)
print("Fetching SPY data...")
spy = ticker.download("SPY", start="2004-01-01", end="2026-03-01")

# 2. FEATURE ENGINEERING & SPLIT
# We use Open, High, Low, Close, Volume as 5 "Channels" for our CNN
data = spy[['Open', 'High', 'Low', 'Close', 'Volume']].copy()
data['Target'] = data['Close'].shift(-1) # Predict next day close
data.dropna(inplace=True)

# Define periods as requested
train_data = data['2004':'2021']
val_data = data['2022':'2024']
test_data = data['2025':'2026']

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_data.drop('Target', axis=1))
val_scaled = scaler.transform(val_data.drop('Target', axis=1))
test_scaled = scaler.transform(test_data.drop('Target', axis=1))

def create_sequences(data, target, window=10):
    X, y = [], []
    for i in range(len(data) - window):
        # Shape: (Channels=5, Width=Window_Size)
        X.append(data[i:i+window].T)
        y.append(target.iloc[i+window])
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(y), dtype=torch.float32)

window_size = 10
X_train, y_train = create_sequences(train_scaled, train_data['Target'], window_size)
X_val, y_val = create_sequences(val_scaled, val_data['Target'], window_size)
X_test, y_test = create_sequences(test_scaled, test_data['Target'], window_size)

# 3. CNN ARCHITECTURE (Inspired by LeNet/Section 7.6)
class FinancialCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Layer 1: Conv (Section 7.4) -> Pooling (Section 7.5)
        self.net = nn.Sequential(
            nn.Conv1d(in_channels=5, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2), # Reduce spatial dimension
            # Layer 2
            nn.Conv1d(16, 32, kernel_size=3),
            nn.ReLU(),
            nn.Flatten(),
            # Fully Connected (Section 7.6)
            nn.LazyLinear(64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

# 4. TRAINING LOGIC
model = FinancialCNN()
loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"\nTraining on 2004-2021 (Samples: {len(X_train)})")
for epoch in range(10):
    model.train()
    optimizer.zero_grad()
    pred = model(X_train).squeeze()
    loss = loss_fn(pred, y_train)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 2 == 0:
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(X_val).squeeze(), y_val)
            print(f"Epoch {epoch+1}: Train Loss {loss.item():.2f}, Val Loss {val_loss.item():.2f}")

# 5. TESTING (2025 - Feb 2026)
model.eval()
with torch.no_grad():
    test_preds = model(X_test).squeeze()
    final_mse = loss_fn(test_preds, y_test)
    print(f"\nFinal Testing MSE (2025-2026): {final_mse.item():.2f}")
    print("Execution complete. The model has processed real SPY data across the 3 requested regimes.")

/tmp/ipykernel_687/632377855.py:10: FutureWarning: YF.download() has changed argument auto_adjust default to True
  spy = ticker.download("SPY", start="2004-01-01", end="2026-03-01")
[*********************100%***********************]  1 of 1 completed

Fetching SPY data...

Training on 2004-2021 (Samples: 4522)



/tmp/ipykernel_687/632377855.py:24: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  train_scaled = scaler.fit_transform(train_data.drop('Target', axis=1))
/tmp/ipykernel_687/632377855.py:25: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  val_scaled = scaler.transform(val_data.drop('Target', axis=1))
/tmp/ipykernel_687/632377855.py:26: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  test_scaled = scaler.transform(test_data.drop('Target', axis=1))


Epoch 2: Train Loss 33658.43, Val Loss 202663.19
Epoch 4: Train Loss 33640.32, Val Loss 202540.41
Epoch 6: Train Loss 33619.03, Val Loss 202400.64
Epoch 8: Train Loss 33594.52, Val Loss 202227.38
Epoch 10: Train Loss 33565.13, Val Loss 202002.92

Final Testing MSE (2025-2026): 394530.00
Execution complete. The model has processed real SPY data across the 3 requested regimes.
